# 03 — Modeling & Evaluation
Train multiple models with MLflow tracking and analyze the best one.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from src.training import ModelTrainer
from src.prediction import ModelPredictor

sns.set_theme(style='whitegrid')
%matplotlib inline

mlflow.set_tracking_uri('../mlruns')

In [ ]:
DATA_PATH = '../data/raw/dataset.csv'
TARGET_COL = 'target'
EXPERIMENT = 'notebook_experiment'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')

In [ ]:
# Training
trainer = ModelTrainer(experiment_name=EXPERIMENT, tracking_uri='../mlruns')
results_df, best_model, preprocessor = trainer.train(df, TARGET_COL, test_size=0.2)

print(f'Task: {preprocessor.task_type}')
print(f'Best model: {type(best_model).__name__}')
results_df

In [ ]:
# Model comparison
if preprocessor.task_type == 'classification':
    plot_metrics = [m for m in ['accuracy', 'balanced_accuracy', 'precision', 'recall',
                                 'f1_weighted', 'f1_macro', 'roc_auc', 'roc_auc_ovr', 'cohen_kappa']
                    if m in results_df.columns]

    df_plot = results_df.set_index('model')[plot_metrics]
    fig, ax = plt.subplots(figsize=(10, 5))
    df_plot.T.plot(kind='bar', ax=ax, colormap='tab10')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Model comparison — classification metrics')
    ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

    if 'log_loss' in results_df.columns:
        fig, ax = plt.subplots(figsize=(6, 3))
        results_df.set_index('model')['log_loss'].plot(kind='barh', ax=ax, color='salmon')
        ax.set_title('Log-Loss (lower is better)')
        plt.tight_layout()
        plt.show()

else:
    error_metrics = [m for m in ['mse', 'rmse', 'mae', 'median_ae'] if m in results_df.columns]
    score_metrics = [m for m in ['r2', 'explained_variance'] if m in results_df.columns]
    if 'mape' in results_df.columns:
        score_metrics.append('mape')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if error_metrics:
        results_df.set_index('model')[error_metrics].plot(kind='bar', ax=axes[0], colormap='tab10')
        axes[0].set_title('Error metrics (lower is better)')
        axes[0].tick_params(axis='x', rotation=30)

    if score_metrics:
        results_df.set_index('model')[score_metrics].plot(kind='bar', ax=axes[1], colormap='Set2')
        axes[1].set_title('Scores (higher is better)')
        axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

In [ ]:
# Radar chart — normalized comparison
if preprocessor.task_type == 'classification':
    radar_metrics = [m for m in ['accuracy', 'balanced_accuracy', 'f1_weighted', 'f1_macro', 'precision', 'recall']
                     if m in results_df.columns]
else:
    radar_metrics = [m for m in ['r2', 'explained_variance'] if m in results_df.columns]

if len(radar_metrics) >= 3:
    import math

    models = results_df['model'].tolist()
    n_metrics = len(radar_metrics)
    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    colors = plt.cm.tab10.colors

    for i, row in results_df.iterrows():
        values = row[radar_metrics].tolist()
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=row['model'], color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_metrics, size=10)
    ax.set_ylim(0, 1)
    ax.set_title('Radar chart — normalized comparison', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature importances
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(
        best_model.feature_importances_,
        index=preprocessor.feature_names_out
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, max(4, len(importances) * 0.3)))
    importances.head(20).plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Feature importances (top 20)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f'{type(best_model).__name__} does not expose feature_importances_')

In [ ]:
# Save model and preprocessor
trainer.save('../models')
print('Model and preprocessor saved to models/')

In [ ]:
# Quick inference test
predictor = ModelPredictor('../models')

sample = df.drop(columns=[TARGET_COL]).head(5)
preds = predictor.predict(sample)
print('Predictions on 5 samples:')
print(preds)

In [ ]:
# Browse MLflow runs
client = mlflow.tracking.MlflowClient('../mlruns')
exp = client.get_experiment_by_name(EXPERIMENT)
if exp:
    order = 'metrics.f1_weighted DESC' if preprocessor.task_type == 'classification' else 'metrics.r2 DESC'
    runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=[order])
    run_rows = [
        {'run_id': r.info.run_id[:8], 'model': r.data.params.get('model'), **r.data.metrics}
        for r in runs
    ]
    pd.DataFrame(run_rows)